# To Bugfix

prepare.py        | steps.py                  | pipeline/                | status
--------------------------------------------------------------------------
prepare_dataset() | download()                | build_manifest()         | X
                  |                           | download_methylation()   | X
                  |                           | initialize_audit_table() | X
                  |                           | build_metadata()         | X
                  |                           | build_biospecimen()      | X
                  |                           | update_metadata()        | X
                  |                           | update_biospecimen()     | X
                  | clean_data()              |                          | X
                  | load_raw_project()        |                          | X
                  | quality_control()         | load_annotation()        | 
                  |                           | sample_qc()              | 
                  |                           | probe_qc()               | 
                  |                           | clean_metadata()         | 
                  | preprocess()              | normalize()              | 
                  |                           | filter_variance()        |
                  |                           | convert_to_mval()        |
                  |                           | impute()                 |
                  | save_project()            |                          | 
prepare_cohort()  | load_processed_project()  |                          |
                  | aggregate_cohort()        | cohort_aggregation()     |
                  | cohort_batch_correction() | batch_correct()          |
                  | aggregate_genes()         | gene_aggregation()       |
                  | winsorize()               |                          |
                  | split()                   |                          |

%load_ext autoreload
%autoreload 2


## Initial Set-up

In [60]:
from methyltrain.config.loader import load_config
from methyltrain.fs.layout import ProjectLayout, CohortLayout

layout = ProjectLayout(
    project_name = "TCGA-CHOL",
    root_dir = "../data",
    raw_dir = "../data/raw/TCGA-CHOL",
    audit_table = "../data/metadata/TCGA-CHOL/TCGA-CHOL_audit_table.csv",
    metadata = "../data/metadata/TCGA-CHOL/TCGA-CHOL_metadata.csv",
    biospecimen = "../data/metadata/TCGA-CHOL/TCGA-CHOL_biospecimen.csv",
    manifest = "../data/metadata/TCGA-CHOL/TCGA-CHOL_manifest.csv",
    status_log = "../data/metadata/TCGA-CHOL/TCGA-CHOL_status_log.csv",
    project_adata = "../data/processed/TCGA-CHOL_adata.h5ad"
)
layout.initialize()
layout.validate()

config = load_config(
    "/Volumes/FBI_Drive/MethylTrain/config/TCGA-CHOL_config_local.yaml"
)

## Current Debug

In [2]:
from methyltrain.api.steps import (
    download, 
    clean_data, 
    quality_control, 
    preprocess, 
    aggregate_cohort, 
    cohort_batch_correction,
    aggregate_genes,
    winsorize,
    split,
    load_raw_project,
    load_processed_project,
)

from methyltrain.utils.load_utils import load_audit_table

%load_ext autoreload
%autoreload 2


In [3]:
import sys
import json
import requests
import subprocess
import time
import datetime

import pandas as pd

from typing import Dict

from methyltrain.fs.layout import ProjectLayout
from methyltrain.constants import GDC_QUERY_URL, MAX_RETRIES, GDC_BIOSPECIMEN_URL
from methyltrain.utils.utils import (
    verify_gdc_client,
    extract_project_id_w_cases,
    extract_project_id_w_project,
    extract_sample_type,
    extract_submitter_id,
    extract_file_id,
    extract_aliquot_id,
    extract_file_name
)
from methyltrain.utils.load_utils import load_status_log

In [ ]:
from methyltrain.pipeline.download import (
    build_manifest, 
    build_metadata,
    download_methylation
)

In [63]:
metadata = build_metadata(audit_table, config)

In [64]:
metadata

,file_name,data_type,data_category,experimental_strategy,platform,project_id,submitter_id,sample_type,aliquot_id,status
file_id,,,,,,,,,,
2494b89c-b222-46a5-a6b5-7bb3caf6bd8f,620834af-09de-42b3-8fae-460228d92c55.methylati...,Methylation Beta Value,DNA Methylation,Methylation Array,Illumina Human Methylation 450,TCGA-CHOL,TCGA-W5-AA39,Primary Tumor,a5cacd63-cf2c-4f53-9bf2-cdaef02b3f34,success
59bfa8f9-8760-468f-bbc2-4b90f030db9b,17f058f2-02f6-4f66-82a4-181030ca941a.methylati...,Methylation Beta Value,DNA Methylation,Methylation Array,Illumina Human Methylation 450,TCGA-CHOL,TCGA-3X-AAVB,Primary Tumor,27eb57d1-20b4-49dd-a091-50a467adb392,success
2e91edbf-e0aa-49dc-885f-d85c8ed3c65a,a5c1aef8-d6e8-414c-bd4e-bde6268e29bd.methylati...,Methylation Beta Value,DNA Methylation,Methylation Array,Illumina Human Methylation 450,TCGA-CHOL,TCGA-W5-AA2I,Primary Tumor,d869cfd9-9fbf-4b10-8101-cecc2a339f12,success
4fcc9b69-14f7-4479-82d1-d5064673bec3,102d6059-eb9f-4510-9f2b-84ceeefe854a.methylati...,Methylation Beta Value,DNA Methylation,Methylation Array,Illumina Human Methylation 450,TCGA-CHOL,TCGA-W5-AA38,Primary Tumor,9b3cf813-5591-4fda-8434-69922528400f,success
46b5f2b0-d534-44c1-8ea6-2d6d441137e5,707e90ec-07eb-4a93-b361-ae2e698d71a5.methylati...,Methylation Beta Value,DNA Methylation,Methylation Array,Illumina Human Methylation 450,TCGA-CHOL,TCGA-W5-AA2G,Primary Tumor,b8ffda8d-1a6e-45dc-b951-0ac021409126,success
6ee17262-7c22-4d22-8713-1dbde299e632,22bf6ab2-ec45-4e59-9425-5f406fb0dd90.methylati...,Methylation Beta Value,DNA Methylation,Methylation Array,Illumina Human Methylation 450,TCGA-CHOL,TCGA-4G-AAZO,Primary Tumor,1ba92c23-02fd-4338-ba81-da6057dfb502,success
e28bede0-b5dc-46ca-8969-f1a61ab2053b,fc69bcce-dbb9-4c7d-a11a-dc51b75e963a.methylati...,Methylation Beta Value,DNA Methylation,Methylation Array,Illumina Human Methylation 450,TCGA-CHOL,TCGA-W5-AA2X,Primary Tumor,dcc3c5fa-2388-401b-b816-6b97d71504b0,success
37e4ad34-94a8-4a7b-8ffc-94a35a0961fb,4b14208f-2f30-4f1a-a4d5-660da0951c95.methylati...,Methylation Beta Value,DNA Methylation,Methylation Array,Illumina Human Methylation 450,TCGA-CHOL,TCGA-W5-AA2Q,Primary Tumor,ae1d0aba-bdda-4192-ba08-cb6e48e9f227,success
3347d5d2-318a-48bf-b0a6-9c4ed4c2b39a,abf98a5a-0d61-4a1a-abe4-b455d8ead3bf.methylati...,Methylation Beta Value,DNA Methylation,Methylation Array,Illumina Human Methylation 450,TCGA-CHOL,TCGA-ZH-A8Y4,Primary Tumor,95240fd1-11cf-4247-9c18-2f01358a5922,success
